# Ранжирование русскоязычных фрагментов

Используется `jinaai/jina-reranker-v3.5`. Для каждого запроса модель получает весь список кандидатов. По представлениям запроса и фрагментов дообучается проектор, который выдаёт скор релевантности.

Финальное обучение проходит на 3 042 запросах и 31 530 парах за две эпохи. Базовый трансформер заморожен. Восстановленный журнал подтверждает 762 обновления проектора. Присланный CSV совпал по SHA-256 с прогнозами второй эпохи.

Запуск выполнялся в Google Colab на Linux, GPU Tesla T4. Данные, кэш и веса сохраняются на Google Drive. Подготовка данных, функция потерь и порядок обучения соответствуют исходному ноутбуку. История экспериментов и результаты приведены в `README.md`.

## 1. Данные

В Colab включите GPU и выполните ячейки по порядку. Подключите Drive. Если CSV отсутствуют, выберите три оригинальных файла соревнования в окне загрузки. Готовые данные и чекпойнты находятся автоматически. Для восстановления прогноза установите `RUN_MODE = "predict"`.

In [ ]:
from pathlib import Path
import hashlib
import re
from google.colab import drive, files

drive.mount("/content/drive")
MY_DRIVE = Path("/content/drive/MyDrive")
WORKING_DIR = MY_DRIVE / "amml_jina_colab"
PROJECT = WORKING_DIR / "amml_jina"
RUN_MODE = "train"  # train — обучение или продолжение; predict — сохранённые веса
PROMPT_TOKEN_CAP = 4096
DATA_OVERRIDE = ""  # папка с тремя CSV на Drive
RESUME_SOURCE = ""  # папка с кэшем прежнего запуска, например amml_jina из Kaggle Output
CHECKPOINT_OVERRIDE = ""  # полный путь на Drive к другому projector.safetensors для predict

CSV_SHA256 = {'train.csv': '2c1b26c5d3ee642f2c959b758cea49d4176ca4c0dfe92fbb7236733eff801042', 'test.csv': '4667c49296b72bf64767338c5345cb2f6c7cfbd0671364e5b728d9d5bcc2bf20', 'sample_submission.csv': 'b2cd50bc4fbaa41cacf04a74d01746c0e9ca391c4e5124822278734700cc3090'}
REQUIRED_CSV = tuple(CSV_SHA256)
assert RUN_MODE in {"train", "predict"}, RUN_MODE
assert RUN_MODE == "predict" or not CHECKPOINT_OVERRIDE, "CHECKPOINT_OVERRIDE используется только в predict."
assert 2048 <= PROMPT_TOKEN_CAP <= 8192, PROMPT_TOKEN_CAP

candidates = [Path(DATA_OVERRIDE)] if DATA_OVERRIDE else [
    WORKING_DIR / "data",
    MY_DRIVE / "amml_bge_2026" / "data",
    Path("/content"),
]
def matching_files(folder):
    if not folder.is_dir():
        return {}
    candidates = sorted(folder.glob("*.csv"))
    found = {}
    for name, digest in CSV_SHA256.items():
        for path in candidates:
            canonical = re.sub(r" \(\d+\)(?=\.csv$)", "", path.name)
            if canonical == name and hashlib.sha256(path.read_bytes()).hexdigest() == digest:
                found[name] = path
                break
    return found


def setup_data(folders, persistent):
    available = {}
    for folder in folders:
        matches = matching_files(folder)
        if len(matches) == len(REQUIRED_CSV) and folder != Path("/content"):
            if all(path.name == name for name, path in matches.items()):
                return folder
        for name, path in matches.items():
            available.setdefault(name, path)
    if DATA_OVERRIDE and len(available) != len(REQUIRED_CSV):
        missing = sorted(set(REQUIRED_CSV) - set(available))
        raise FileNotFoundError(f"Нужны оригинальные файлы в {DATA_OVERRIDE}: {missing}")
    if len(available) != len(REQUIRED_CSV):
        print("Выберите оригинальные файлы:", ", ".join(name for name in REQUIRED_CSV if name not in available))
        uploaded = files.upload()
        for filename, content in uploaded.items():
            name = re.sub(r" \(\d+\)(?=\.csv$)", "", Path(filename).name)
            if name in CSV_SHA256 and hashlib.sha256(content).hexdigest() == CSV_SHA256[name]:
                available.setdefault(name, content)

    if len(available) != len(REQUIRED_CSV):
        missing = sorted(set(REQUIRED_CSV) - set(available))
        raise ValueError(f"Файлы отсутствуют или отличаются от оригиналов конкурса: {missing}")

    persistent.mkdir(parents=True, exist_ok=True)
    for name, source in available.items():
        target = persistent / name
        if target == source:
            continue
        if target.exists():
            if hashlib.sha256(target.read_bytes()).hexdigest() == CSV_SHA256[name]:
                continue
            raise FileExistsError(f"В {target} лежит другой файл. Переместите его или задайте DATA_OVERRIDE.")
        temporary = target.with_suffix(".uploading")
        temporary.write_bytes(source.read_bytes() if isinstance(source, Path) else source)
        temporary.replace(target)
    return persistent


DATA_DIR = setup_data(candidates, WORKING_DIR / "data")
PROJECT.mkdir(parents=True, exist_ok=True)
print("Данные:", DATA_DIR)
print("Google Drive:", WORKING_DIR)


## 2. Зависимости

Устанавливаются те же версии библиотек для загрузки Jina, что в исходном запуске. CUDA-версия PyTorch берётся из среды Colab. После запуска фактические версии пакетов сохраняются рядом с CSV в `requirements.txt`.

In [ ]:
import subprocess
import sys
import os
import torch
from importlib.metadata import version, PackageNotFoundError
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HUB_DISABLE_XET']='1'
versions={'transformers':'4.57.3','tokenizers':'0.22.1','huggingface-hub':'0.36.2','safetensors':'0.8.0'}
def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None
print("Вычислитель:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
pins = [f"{name}=={ver}" for name, ver in versions.items() if installed(name) != ver]
if pins:
    proc=subprocess.run([sys.executable,'-m','pip','install',*pins],capture_output=True,text=True)
    (PROJECT / 'jina_install.log').write_text(proc.stdout+proc.stderr)
    if proc.returncode:
        print(proc.stdout+proc.stderr)
        raise RuntimeError('Установка зависимостей остановилась.')
stale=[name for name,expected in versions.items() if name.replace('-','_') in sys.modules
       and getattr(sys.modules[name.replace('-','_')],'__version__',expected)!=expected]
if stale:
    raise RuntimeError('Перезапустите среду после установки: '+', '.join(stale))


In [ ]:
import copy

import hashlib

import html

import importlib

import json

import math

import os

import re

import time

import unicodedata

from pathlib import Path

from functools import lru_cache

import numpy as np

import pandas as pd

import torch

import torch.nn.functional as F

from safetensors.torch import load_file, save_file

from sklearn.model_selection import GroupShuffleSplit

from tqdm.auto import tqdm

MODEL = 'jinaai/jina-reranker-v3.5'

REVISION = 'e8a93f33f0b22108f8c2364f8484ce3422552fbc'

EXPECTED_DATA = '4f9e3fd0fa25e2c350fa9b8f7f925f6fd4d9ec34d610a8e001c76b321db5f556'

SEED = 2027

PILOT_QUERIES = 384

SHARD_QUERIES = 16

QUERY_CAP = 96

DOCUMENT_CAP = 512

TOTAL_CAP = 4096

SCORE_SCALE = 16.0

CODE_SHA = '9379e7d9b2a783ee335fbe3135881b8c774820e0ddfa0e582bda62fbe9109eac'


## Данные и разбиение

In [ ]:
def text_clean(value):
    s = html.unescape(html.unescape('' if pd.isna(value) else str(value)))
    s = re.sub('<[^>]+>', ' ', s)
    s = unicodedata.normalize('NFC', unicodedata.normalize('NFD', s).replace('́', ''))
    s = re.sub('(?m)^\\s*(?:Навигация:.*|Главная [›»].*|Вернуться к началу страницы|Все материалы сайта)\\s*$', ' ', s)
    return re.sub('\\s+', ' ', s).strip()

def read_inputs(data_dir):
    data_dir = Path(data_dir)
    frames = [pd.read_csv(data_dir / name, dtype={'id': str, 'query_id': str, 'article_id': str}) for name in ['train.csv', 'test.csv', 'sample_submission.csv']]
    train, test, sample = frames
    required = {'id', 'query_id', 'article_id', 'query', 'title', 'text'}
    assert required <= set(train) and required <= set(test), 'Не хватает текстовых колонок'
    assert train.id.is_unique and test.id.is_unique and sample.id.is_unique
    assert not set(train.id) & set(test.id)
    assert not set(train.query_id) & set(test.query_id)
    assert set(train.relevance.unique()) <= {0, 1}
    assert set(test.id) == set(sample.id), 'sample_submission и test имеют разные id'
    assert sample.columns.tolist() == ['id', 'prediction']
    for frame in [train, test]:
        assert not frame[['id', 'query_id', 'query', 'text']].isna().any().any()
        for c in ['query', 'title', 'text']:
            frame[c] = frame[c].map(text_clean)
    hasher = hashlib.sha256()
    for name in ['train.csv', 'test.csv', 'sample_submission.csv']:
        hasher.update(name.encode())
        with open(data_dir / name, 'rb') as f:
            for block in iter(lambda: f.read(1024 * 1024), b''):
                hasher.update(block)
    return (train, test, sample, hasher.hexdigest())

def query_splits(frame):
    train_rows, valid_rows = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(frame, groups=frame.query_id))
    train, valid = (frame.iloc[train_rows], frame.iloc[valid_rows])
    qids = np.array(sorted(valid.query_id.unique()))
    np.random.default_rng(SEED).shuffle(qids)
    dev = valid.query_id.isin(qids[:len(qids) // 2])

    def normalized(text):
        return ' '.join(re.findall('\\w+', text.casefold().replace('ё', 'е')))
    heldout = set(valid['query'].map(normalized))
    repeated = train['query'].map(normalized).isin(heldout)
    excluded = set(train.loc[repeated, 'query_id'])
    train = train.loc[~train.query_id.isin(excluded)]
    assert (train.query_id.nunique(), dev.groupby(valid.query_id).first().sum(), valid.loc[~dev].query_id.nunique()) == (2431, 304, 305)
    assert not set(train.query_id) & set(valid.query_id)
    return (train, valid.loc[dev], valid.loc[~dev], len(excluded))

def grouped(frame):
    return [(str(query), group.sort_values('id')) for query, group in frame.groupby('query_id', sort=True)]

def pick_pilot(frame):
    queries = np.array(sorted(frame.query_id.unique()))
    selected = np.random.default_rng(SEED).choice(queries, PILOT_QUERIES, replace=False)
    return frame[frame.query_id.isin(selected)]

def all_train_parts(train):
    fit, dev, audit, _ = query_splits(train)
    pilot = pick_pilot(fit)
    remaining = fit[~fit.query_id.isin(pilot.query_id)]
    used = set(fit.query_id) | set(dev.query_id) | set(audit.query_id)
    previously_excluded = train[~train.query_id.isin(used)]
    parts = (('pilot_train', pilot), ('other_train', remaining), ('dev', dev), ('audit', audit), ('previously_excluded', previously_excluded))
    assert [part.query_id.nunique() for _, part in parts] == [384, 2047, 304, 305, 2]
    rows = pd.concat([part['id'] for _, part in parts])
    assert len(rows) == len(train) == 31530
    assert rows.is_unique and set(rows) == set(train.id)
    assert train.query_id.nunique() == 3042
    return parts

def permute(query_id, count):
    seed = int.from_bytes(hashlib.sha256(f'{SEED}:{query_id}'.encode()).digest()[:8], 'little')
    return np.random.default_rng(seed).permutation(count)


## Представления Jina

In [ ]:
def new_reranker(device, location=MODEL):
    from transformers import AutoModel, AutoTokenizer
    params = {'revision': REVISION} if location == MODEL else {}
    dtype = torch.float16 if device.type == 'cuda' else torch.float32
    model = AutoModel.from_pretrained(location, **params, trust_remote_code=True, use_safetensors=True, torch_dtype=dtype, attn_implementation='sdpa').to(device).eval()
    tokenizer = AutoTokenizer.from_pretrained(location, **params, trust_remote_code=False)
    return (model, tokenizer)

def shortened(text, tokenizer, limit, encode_text=None):
    tokens = tokenizer.encode(text, add_special_tokens=False) if encode_text is None else encode_text(text)
    if len(tokens) <= limit:
        return (text, False)
    head = (limit - 5) * 3 // 4
    tail = limit - 5 - head
    result = tokenizer.decode(tokens[:head]) + '\n[…]\n' + tokenizer.decode(tokens[-tail:])
    return (result, True)

def make_prompt(model, tokenizer, frame, order, cap, shorten=None):
    if shorten is None:
        shorten = lambda text, limit: shortened(text, tokenizer, limit)
    query = frame['query'].iloc[0]
    tokens = tokenizer.all_special_tokens + list(model.special_tokens.values())

    def escape(text):
        for token in tokens:
            text = text.replace(token, ' ')
        return text
    raw = [escape(row.title + '\n' + row.text) for row in frame.itertuples()]
    q, _ = shorten(escape(query), QUERY_CAP)
    fmt = importlib.import_module(type(model).__module__).format_docs_prompts_func
    doc_cap = min(DOCUMENT_CAP, max(48, (cap - 200) // len(raw)))
    while True:
        docs, flags = zip(*(shorten(raw[int(i)], doc_cap) for i in order))
        prompt = fmt(q, list(docs), special_tokens=model.special_tokens)
        batch = tokenizer(prompt, return_tensors='pt', add_special_tokens=False)
        length = batch['input_ids'].shape[-1]
        if length <= cap:
            return (batch, length, sum(flags))
        if doc_cap <= 48:
            raise ValueError(f'Невозможно разместить все {len(raw)} кандидатов в {cap} токенах.')
        doc_cap = max(48, min(doc_cap - 16, doc_cap - math.ceil((length - cap) / len(raw)) - 8))

def encode_view(model, tokenizer, frame, order, cap, shorten=None):
    batch, length, truncated = make_prompt(model, tokenizer, frame, order, cap, shorten)
    device = next(model.parameters()).device
    batch = {k: v.to(device) for k, v in batch.items()}
    ids = batch['input_ids'][0]
    qpos = (ids == model.query_embed_token_id).nonzero().flatten()
    dpos = (ids == model.doc_embed_token_id).nonzero().flatten()
    assert (qpos.numel(), dpos.numel()) == (1, len(frame))
    assert int(qpos[0]) > int(dpos[-1]), 'Jina query marker must follow all documents'
    with torch.inference_mode():
        h = model.model(**batch, use_cache=False, return_dict=True).last_hidden_state[0]
        docs = h[dpos][torch.as_tensor(np.argsort(order), device=device)]
        vectors = torch.cat([h[qpos], docs], dim=0).cpu().numpy().astype(np.float16)
    assert np.isfinite(vectors).all()
    return (vectors, length, truncated)

def encode_group(model, tokenizer, query_id, frame):
    order = permute(query_id, len(frame))

    @lru_cache(maxsize=None)
    def encode_text(text):
        return tokenizer.encode(text, add_special_tokens=False)

    @lru_cache(maxsize=None)
    def shorten(text, limit):
        return shortened(text, tokenizer, limit, encode_text)
    results = [encode_view(model, tokenizer, frame, view, TOTAL_CAP, shorten) for view in (order, order[::-1].copy())]
    vectors = np.stack([item[0] for item in results])
    return (vectors, max((item[1] for item in results)), max((item[2] for item in results)))

def save_shard(path, records):
    sizes = np.array([x[0].shape[1] for x in records], dtype=np.int32)
    vectors = np.concatenate([x[0] for x in records], axis=1)
    temporary = path.with_suffix('.tmp')
    with temporary.open('wb') as stream:
        np.savez(stream, vectors=vectors, sizes=sizes, stats=np.asarray([(x[1], x[2]) for x in records]))
    os.replace(temporary, path)

def load_shard(path, expected):
    if not path.exists():
        return []
    with np.load(path, allow_pickle=False) as f:
        sizes = f['sizes']
        vectors = f['vectors']
        stats = f['stats']
    assert len(sizes) <= len(expected) and np.array_equal(sizes, expected[:len(sizes)])
    assert vectors.shape == (2, int(sizes.sum()), 1024) and np.isfinite(vectors).all()
    chunks = np.split(vectors, np.cumsum(sizes)[:-1], axis=1)
    return [(v, int(pair[0]), int(pair[1])) for v, pair in zip(chunks, stats)]

def collect_features(entries, root, provider):
    root.mkdir(parents=True, exist_ok=True)
    identity = hashlib.sha256(json.dumps([[q, g.id.tolist()] for q, g in entries], ensure_ascii=False).encode()).hexdigest()
    manifest = root / 'manifest.json'
    if manifest.exists():
        assert json.loads(manifest.read_text())['ids_digest'] == identity, 'Изменился состав кэша.'
    else:
        save_json(manifest, {'ids_digest': identity, 'groups': len(entries), 'revision': REVISION})
    result = []
    new = 0
    started = time.monotonic()
    with tqdm(total=len(entries), desc=root.name) as progress:
        for start in range(0, len(entries), SHARD_QUERIES):
            path = root / f'shard_{start:05d}.npz'
            block = entries[start:start + SHARD_QUERIES]
            records = load_shard(path, np.asarray([len(g) + 1 for _, g in block]))
            original_count = len(records)
            progress.update(original_count)
            try:
                for query_id, frame in block[original_count:]:
                    model, tokenizer = provider()
                    records.append(encode_group(model, tokenizer, query_id, frame))
                    new += 1
                    progress.update(1)
            finally:
                if len(records) > original_count:
                    save_shard(path, records)
            result.extend(records)
    return (result, {'lists': len(entries), 'new_lists': new, 'seconds_this_call': time.monotonic() - started, 'max_prompt_tokens': max((x[1] for x in result)), 'lists_with_truncation': sum((x[2] > 0 for x in result))})

def load_assets(project):
    from huggingface_hub import snapshot_download
    root = Path(project) / 'weights' / REVISION[:12]
    ready = all(((root / name).is_file() for name in ['config.json', 'modeling.py', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']))
    if not ready:
        snapshot_download(MODEL, revision=REVISION, local_dir=root, allow_patterns=['*.py', '*.json', '*.safetensors', '*.txt'], max_workers=3)
    return root

def original_projector(assets):
    from safetensors import safe_open
    head = torch.nn.Sequential(torch.nn.Linear(1024, 512, bias=False), torch.nn.ReLU(), torch.nn.Linear(512, 512, bias=False))
    with safe_open(str(assets / 'model.safetensors'), framework='pt', device='cpu') as stream:
        head.load_state_dict({name: stream.get_tensor('projector.' + name) for name in head.state_dict()})
    return head.float().eval()

def runtime_provider(assets):
    cache = []

    def get():
        if not cache:
            if not torch.cuda.is_available():
                raise RuntimeError('Для расчёта недостающих представлений включите GPU. Сохранённый кэш можно обработать на CPU.')
            torch.cuda.reset_peak_memory_stats()
            cache.extend(new_reranker(torch.device('cuda'), str(assets)))
        return cache
    return get


## Дообучение проектора

In [ ]:
def ranked_scores(projector, vectors):
    embeds = F.normalize(projector(vectors.float()), dim=-1)
    return (embeds[:, 1:] * embeds[:, :1]).sum(dim=-1)

def score_entries(projector, frame, entries, features):
    by_view = [{}, {}]
    projector.eval()
    with torch.inference_mode():
        for (_, group), (vectors, _, _) in zip(entries, features, strict=True):
            scores = ranked_scores(projector, torch.from_numpy(vectors)).numpy()
            for i in range(2):
                by_view[i].update(zip(group.id, scores[i], strict=True))
    values = np.stack([frame.id.map(s).to_numpy(float) for s in by_view])
    assert np.isfinite(values).all()
    return values

def load_projector(base, path):
    projector = copy.deepcopy(base).float().cpu()
    projector.load_state_dict(load_file(str(path), device='cpu'), strict=True)
    if not all((torch.isfinite(parameter).all().item() for parameter in projector.parameters())):
        raise ValueError(f'Некорректные значения проектора: {path}')
    return projector.eval()

def training_signature(entries):
    return hashlib.sha256(json.dumps([[query, frame.id.tolist()] for query, frame in entries], ensure_ascii=False).encode()).hexdigest()

def projector_checksum(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def read_trained_projector(base, root, signature):
    path = root / 'projector.safetensors'
    first = root / 'projector_epoch1.safetensors'
    metadata = root / 'projector_training.json'
    if not (path.is_file() and first.is_file() and metadata.is_file()):
        return None
    report = json.loads(metadata.read_text())
    if report.get('epochs') == 2 and report.get('train_signature') == signature and (report.get('checkpoint_sha256') == projector_checksum(path)) and (report.get('epoch1_sha256') == projector_checksum(first)):
        return (load_projector(base, path), {**report, 'reused_checkpoint': True})
    return None

def fit_projector(base, entries, features, root):
    first = root / 'projector_epoch1.safetensors'
    path = root / 'projector.safetensors'
    metadata = root / 'projector_training.json'
    resume = root / 'optimizer_epoch1.pt'
    signature = training_signature(entries)
    saved_projector = read_trained_projector(base, root, signature)
    if saved_projector is not None:
        return saved_projector
    projector = copy.deepcopy(base).float().cpu().train()
    opt = torch.optim.AdamW(projector.parameters(), lr=1e-05, weight_decay=0.01)
    batch_size = 8
    updates_per_epoch = math.ceil(len(entries) / batch_size)
    completed = 0
    start_epoch = 1
    started = time.monotonic()
    if first.is_file() and resume.is_file():
        saved = torch.load(resume, map_location='cpu', weights_only=True)
        if saved['train_signature'] == signature and saved['projector_sha256'] == projector_checksum(first):
            projector.load_state_dict(load_file(str(first), device='cpu'), strict=True)
            opt.load_state_dict(saved['optimizer'])
            start_epoch = 2
            completed = updates_per_epoch
            print('Восстановлена первая эпоха и состояние AdamW. Продолжаем со второй.')
    for epoch in range(start_epoch, 3):
        order = np.random.default_rng(SEED + epoch - 1).permutation(len(entries))
        for start in tqdm(range(0, len(order), batch_size), desc=f'Проектор на CPU, эпоха {epoch}/2'):
            block = order[start:start + batch_size]
            opt.zero_grad(set_to_none=True)
            for i in block:
                group = entries[i][1]
                vectors = torch.from_numpy(features[i][0])
                scores = ranked_scores(projector, vectors) * SCORE_SCALE
                mask = group.relevance.eq(1).to_numpy()
                labels = torch.tensor(mask, dtype=torch.float32).expand_as(scores)
                bce = F.binary_cross_entropy_with_logits(scores, labels)
                if mask.any() and (~mask).any():
                    articles = group.article_id.to_numpy()
                    same = articles[mask, None] == articles[None, ~mask]
                    weight = 1.0 + 0.5 * torch.tensor(same, dtype=torch.float32)
                    relative = F.softplus(scores[:, None, ~mask] - scores[:, mask, None])
                    loss = 0.85 * (relative * weight).mean() / weight.mean() + 0.15 * bce
                else:
                    loss = bce
                assert torch.isfinite(loss)
                (loss / len(block)).backward()
            norm = torch.nn.utils.clip_grad_norm_(projector.parameters(), 1.0)
            assert torch.isfinite(norm)
            opt.step()
            completed += 1
        checkpoint = first if epoch == 1 else path
        temp = checkpoint.with_suffix('.tmp')
        save_file({key: value.detach().contiguous() for key, value in projector.state_dict().items()}, str(temp))
        temp.replace(checkpoint)
        if epoch == 1:
            temp_resume = resume.with_suffix('.tmp')
            torch.save({'train_signature': signature, 'projector_sha256': projector_checksum(first), 'optimizer': opt.state_dict()}, temp_resume)
            temp_resume.replace(resume)
        print(f'Чекпойнт эпохи {epoch} на Drive: {checkpoint}', flush=True)
    assert any((not torch.equal(a, b) for a, b in zip(base.parameters(), projector.parameters(), strict=True)))
    reloaded = load_projector(base, path)
    with torch.inference_mode():
        check = torch.from_numpy(features[0][0])
        assert torch.equal(ranked_scores(projector.eval(), check), ranked_scores(reloaded, check))
    report = {'queries': len(entries), 'epochs': 2, 'updates_per_epoch': updates_per_epoch, 'successful_updates': completed, 'seconds_cpu_this_session': time.monotonic() - started, 'checkpoint': path.name, 'checkpoint_sha256': projector_checksum(path), 'epoch1_checkpoint': first.name, 'epoch1_sha256': projector_checksum(first), 'train_signature': signature}
    save_json(metadata, report)
    return (reloaded, report)


## Сохранение результата

In [ ]:
def write_submission(test, sample, prediction, path):
    assert len(prediction) == len(test) and np.isfinite(prediction).all()
    lookup = pd.Series(np.asarray(prediction, dtype=np.float64), index=test.id)
    result = pd.DataFrame({'id': sample.id, 'prediction': sample.id.map(lookup)})
    assert result.id.equals(sample.id) and result.id.is_unique and result.prediction.notna().all()
    result.to_csv(path, index=False)
    print('Сохранено', path, 'строк:', len(result))
    return result

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix('.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str))
    os.replace(temp, path)

def runtime_requirements():
    import importlib.metadata as metadata
    from packaging.requirements import Requirement
    packages = {}

    def visit(name):
        dist = metadata.distribution(name)
        key = dist.metadata['Name']
        if key.lower() in packages:
            return
        packages[key.lower()] = (key, dist.version)
        for raw in dist.requires or []:
            dep = Requirement(raw)
            if dep.marker is None or dep.marker.evaluate({'extra': ''}):
                visit(dep.name)
    for name in ['torch', 'transformers', 'numpy', 'pandas', 'scikit-learn', 'nbformat', 'tqdm', 'safetensors']:
        visit(name)
    prefix = ''
    torch_version = metadata.version('torch')
    if '+cpu' in torch_version:
        prefix = '--extra-index-url https://download.pytorch.org/whl/cpu\n'
    if '+cu' in torch_version:
        tag = torch_version.split('+')[-1]
        prefix = '--extra-index-url https://download.pytorch.org/whl/' + tag + '\n'
    return prefix + '\n'.join((f'{name}=={version}' for name, version in sorted(packages.values(), key=lambda x: x[0].lower()))) + '\n'


## 3. Обучение и прогноз

Кэш разбит на группы от прежних запусков для повторного использования вычислений. Все пять групп входят в итоговое обучение. Метки прежних dev и audit теперь используются при обучении, поэтому независимая метрика на них здесь не рассчитывается.

После двух эпох сохраняются проекторы и два CSV. `submission.csv` соответствует второй эпохе, `submission_epoch1.csv` — первой. Перед сохранением проверяются столбцы, число строк, ID и конечность прогнозов.

In [ ]:
from shutil import copytree, rmtree
from zipfile import ZIP_DEFLATED, ZipFile
TOTAL_CAP = PROMPT_TOKEN_CAP
RUN_KEY = f'{EXPECTED_DATA[:16]}_{CODE_SHA[:10]}_cap{TOTAL_CAP}'
RUN_ROOT = PROJECT / RUN_KEY
OUTPUT_DIR = RUN_ROOT / 'full_train_3042_2epochs'
CHECKPOINT = Path(CHECKPOINT_OVERRIDE) if CHECKPOINT_OVERRIDE else OUTPUT_DIR / 'projector.safetensors'
if CHECKPOINT_OVERRIDE and (not CHECKPOINT.is_file()):
    raise FileNotFoundError(f'Файл проектора не найден: {CHECKPOINT}')
RESULT_DIR = WORKING_DIR / 'full_train_3042_2epochs'
if CHECKPOINT_OVERRIDE:
    checkpoint_hash = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()[:12]
    RESULT_DIR = RESULT_DIR / 'predictions' / checkpoint_hash
RESULT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION = RESULT_DIR / 'submission.csv'

def restore_cache():
    if (RUN_ROOT / 'cache').is_dir():
        print('Текущий кэш:', RUN_ROOT / 'cache')
        return
    if RESUME_SOURCE:
        source = Path(RESUME_SOURCE) / RUN_KEY
        if not (source / 'cache').is_dir():
            raise FileNotFoundError(f'В {source} нет совместимого кэша.')
    else:
        previous = MY_DRIVE / 'amml_bge_2026' / 'jina_listwise' / RUN_KEY
        candidates = [previous] if (previous / 'cache').is_dir() else []
        if len(candidates) != 1:
            if candidates:
                print('Найдено несколько кэшей. Для выбора задайте RESUME_SOURCE:', candidates)
            return
        source = candidates[0]
    temporary = RUN_ROOT.with_name(RUN_ROOT.name + '.restore')
    if temporary.exists():
        rmtree(temporary)
    copytree(source, temporary, ignore=lambda _, names: [n for n in names if n.endswith('.tmp')])
    if RUN_ROOT.exists():
        copytree(temporary, RUN_ROOT, dirs_exist_ok=True)
        rmtree(temporary)
    else:
        temporary.replace(RUN_ROOT)
    print('Кэш восстановлен из:', source)

def verify_submission(path, sample_path):
    expected = pd.read_csv(sample_path, dtype={'id': str}, keep_default_na=False)
    saved = pd.read_csv(path, dtype={'id': str}, keep_default_na=False)
    if saved.columns.tolist() != ['id', 'prediction']:
        raise ValueError('В submission.csv должны быть ровно два столбца: id,prediction.')
    if len(saved) != 13507 or not saved['id'].equals(expected['id']) or (not saved['id'].is_unique):
        raise ValueError('Набор и порядок ID должны совпадать с 13 507 строками sample_submission.csv.')
    values = pd.to_numeric(saved['prediction'], errors='coerce').to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('В столбце prediction есть пропуски или некорректные значения.')
    data = path.read_bytes()
    return {'file': path.name, 'columns': saved.columns.tolist(), 'rows': len(saved), 'bytes': len(data), 'sha256': hashlib.sha256(data).hexdigest()}

def predict_full():
    train, test, sample, digest = read_inputs(DATA_DIR)
    assert digest == EXPECTED_DATA, 'Нужны оригинальные CSV соревнования'
    restore_cache()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if RUN_MODE == 'predict' and (not CHECKPOINT.is_file()):
        raise FileNotFoundError(f'Проектор не найден: {CHECKPOINT}. Укажите CHECKPOINT_OVERRIDE или запустите train.')
    report_path = OUTPUT_DIR / 'report.json' if RUN_MODE == 'train' else RESULT_DIR / 'prediction_report.json'
    report = {'stage': 'full', 'status': 'running', 'model': MODEL, 'revision': REVISION, 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', 'run_mode': RUN_MODE, 'data_sha256': digest, 'training_queries': train.query_id.nunique() if RUN_MODE == 'train' else None, 'training_pairs': len(train) if RUN_MODE == 'train' else None, 'test_queries': test.query_id.nunique(), 'total_cap': TOTAL_CAP}
    if report_path.is_file():
        previous = report_path.with_name('report_previous.json')
        previous.write_bytes(report_path.read_bytes())
    save_json(report_path, report)
    started = time.monotonic()
    torch.set_num_threads(min(4, os.cpu_count() or 1))
    assets = load_assets(PROJECT)
    print('Базовые веса Jina на Drive:', assets)
    base_head = original_projector(assets)
    provider = runtime_provider(assets)

    def features(name, frame):
        entries = grouped(frame)
        vectors, timing = collect_features(entries, RUN_ROOT / 'cache' / name, provider)
        report[name + '_encoding'] = timing
        save_json(report_path, report)
        return (entries, vectors)
    if RUN_MODE == 'train':
        parts = all_train_parts(train)
        entries = [group for _, frame in parts for group in grouped(frame)]
        assert len(entries) == 3042
        saved = read_trained_projector(base_head, OUTPUT_DIR, training_signature(entries))
        if saved is not None:
            tuned, report['training'] = saved
            print('Проверены оба чекпойнта. Признаки train повторно не загружаются.')
        else:
            vectors = []
            for name, frame in parts:
                _, values = features(name, frame)
                vectors.extend(values)
            tuned, report['training'] = fit_projector(base_head, entries, vectors, OUTPUT_DIR)
            del vectors, values
        del entries, parts
    else:
        tuned = copy.deepcopy(base_head)
        tuned.load_state_dict(load_file(str(CHECKPOINT), device='cpu'), strict=True)
        if not all((torch.isfinite(parameter).all().item() for parameter in tuned.parameters())):
            raise ValueError(f'Некорректные значения в проекторе: {CHECKPOINT}')
        tuned.eval()
    report['checkpoint'] = str(CHECKPOINT)
    report['checkpoint_sha256'] = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
    save_json(report_path, report)
    print('Проектор на Drive:', CHECKPOINT, flush=True)
    e, v = features('test', test)

    def save_predictions(projector, path, receipt_path):
        scores = score_entries(projector, test, e, v).mean(axis=0)
        temporary_csv = path.with_suffix('.tmp')
        write_submission(test, sample, scores, temporary_csv)
        receipt = verify_submission(temporary_csv, DATA_DIR / 'sample_submission.csv')
        temporary_csv.replace(path)
        receipt['file'] = path.name
        save_json(receipt_path, receipt)
        print(f"ПРОВЕРЕНО: {path}, {receipt['rows']:,} строк.", flush=True)
    save_predictions(tuned, SUBMISSION, RESULT_DIR / 'submission_check.json')
    if RUN_MODE == 'train':
        first = load_projector(base_head, OUTPUT_DIR / 'projector_epoch1.safetensors')
        epoch1_csv = RESULT_DIR / 'submission_epoch1.csv'
        save_predictions(first, epoch1_csv, RESULT_DIR / 'submission_epoch1_check.json')
        report['submission_epoch1'] = str(epoch1_csv)
    del e, v
    report.update(status='submission_ready', submission=str(SUBMISSION), submission_rows=len(sample))
    save_json(report_path, report)
    print('CSV записан:', SUBMISSION, flush=True)
    report['seconds_this_run'] = time.monotonic() - started
    report['peak_gpu_gib'] = round(torch.cuda.max_memory_allocated() / 2 ** 30, 3) if torch.cuda.is_available() else 0
    save_json(report_path, report)
    environment = {'os': sys.platform, 'python': sys.version, 'torch': torch.__version__, 'cuda': torch.version.cuda, 'device': report['device']}
    report['environment'] = environment
    save_json(report_path, report)
    try:
        (RESULT_DIR / 'requirements.txt').write_text(runtime_requirements(), encoding='utf-8')
    except Exception as error:
        print('Не удалось сохранить версии пакетов:', error)
    print('Среда выполнения:', json.dumps(environment, ensure_ascii=False))
    print('CSV для Kaggle:', SUBMISSION)
    print('Веса и отчёт:', OUTPUT_DIR)
    return report
REPORT = predict_full()


## 4. Скачать результат

Для Kaggle нужен `submission.csv`. Он также остаётся в папке `Мой диск/amml_jina_colab/full_train_3042_2epochs`. Эту ячейку можно выполнить отдельно в новой сессии, чтобы скачать уже готовый файл.

In [ ]:
from pathlib import Path
from google.colab import drive, files

mount_point = Path("/content/drive")
if not (mount_point / "MyDrive").is_dir():
    drive.mount(str(mount_point))

saved_csv = mount_point / "MyDrive/amml_jina_colab/full_train_3042_2epochs/submission.csv"
path = Path(globals().get("SUBMISSION", saved_csv))
if not path.is_file():
    raise FileNotFoundError(path)
print("Файл отправки:", path)
files.download(str(path))
